In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 23
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.68136693075172

Trial 1 =========================================
16.07735945841761

Trial 2 =========================================
12.976971845456179

Trial 3 =========================================
14.507423914023517

Trial 4 =========================================
14.811583437402803

Trial 5 =========================================
15.341283467121396

Trial 6 =========================================
14.665793650713919

Trial 7 =========================================
14.15302531884271

Trial 8 =========================================
13.894724416611538

Trial 9 =========================================
13.200662921811775

Trial 10 =========================================
13.13444167224118

Trial 11 =========================================
14.667700487447796

Trial 12 =========================================
15.211423917207583

Trial 13 =========================================
14.45677672019034

Trial 14 ============

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 35 =========================================
15.533942467350524

Trial 36 =========================================
13.065389126128299

Trial 37 =========================================
15.748431080767963

Trial 38 =========================================
16.170724856390848

Trial 39 =========================================
15.133560369691338

Trial 40 =========================================
14.150063197061769

Trial 41 =========================================
14.041380231670086

Trial 42 =========================================
13.657573826568342

Trial 43 =========================================
15.060696086864837

Trial 44 =========================================
16.258518610677633

Trial 45 =========================================
14.701579872581675

Trial 46 =========================================
14.810765740930561

Trial 47 =========================================
15.013679276397024

Trial 48 =========================================
15.911991060160034

Trial 

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 59 =========================================
15.883741698806531

Trial 60 =========================================
14.415796075822856

Trial 61 =========================================
14.290857786540768

Trial 62 =========================================
16.337466735319715

Trial 63 =========================================
15.234802885643802

Trial 64 =========================================
13.623719011149444

Trial 65 =========================================
16.525211588861055

Trial 66 =========================================
12.68048013288733

Trial 67 =========================================
14.027483217971728

Trial 68 =========================================
13.912972882971246

Trial 69 =========================================
14.693241805719733

Trial 70 =========================================
13.453017891380968

Trial 71 =========================================
13.639143609035298

Trial 72 =========================================
15.351250592429981

Trial 7

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 76 =========================================
15.975814099311414

Trial 77 =========================================
13.291867057420516

Trial 78 =========================================
13.525987209994032

Trial 79 =========================================
14.389559343854827

Trial 80 =========================================
14.450895416578692

Trial 81 =========================================
16.455229682942257

Trial 82 =========================================
13.449890889325946

Trial 83 =========================================
14.557819787678579

Trial 84 =========================================
13.515847313724198

Trial 85 =========================================
13.784206372043121

Trial 86 =========================================
13.351020550231164

Trial 87 =========================================
14.874360471076496

Trial 88 =========================================
14.162348315404339

Trial 89 =========================================
13.551588378840172

Trial 

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.0805432984015
Avg = 14.624558348212728
Std = 1.0700459796844644


In [6]:
print(y_max_arr.tolist())

[13.68136693075172, 16.07735945841761, 12.976971845456179, 14.507423914023517, 14.811583437402803, 15.341283467121396, 14.665793650713919, 14.15302531884271, 13.894724416611538, 13.200662921811775, 13.13444167224118, 14.667700487447796, 15.211423917207583, 14.45677672019034, 15.660377864446577, 13.676601773684434, 15.095699163711291, 13.257000839566258, 16.027904084060367, 15.775689744225987, 14.730904159904538, 14.556590244019098, 15.070748865973913, 17.58928161558243, 13.960642408306805, 14.00843110022026, 14.814867935080338, 13.449448057705933, 15.441353703679289, 15.137938221863065, 14.288638240039218, 13.439954032075022, 13.270141340342823, 13.600984398189341, 13.632335694813799, 15.533942467350524, 13.065389126128299, 15.748431080767963, 16.170724856390848, 15.133560369691338, 14.150063197061769, 14.041380231670086, 13.657573826568342, 15.060696086864837, 16.258518610677633, 14.701579872581675, 14.810765740930561, 15.013679276397024, 15.911991060160034, 14.396226171643141, 14.222

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-23.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)